# 10 — CNN classifier: lens vs no-lens

Train a small VGG-like CNN to distinguish galaxies that show a strong-lens arc/ring from plain galaxies. Both classes are synthesized from the package's forward models so we have an unlimited supply of perfectly-labelled training data.

In [ ]:
# Bootstrap: make `lensing` importable when running notebooks/ directly.
import sys
from pathlib import Path
repo = Path.cwd().resolve().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import lensing as gl
# Device-agnostic: prefer MPS (Apple GPU) → CUDA → CPU.
# Pass device="cpu" if you need to force the CPU path (e.g. for
# operators that have no MPS kernel yet, or for reproducibility).
device, dtype = gl.config.setup(seed=42)
print(f"using device: {device}")


## 1. Synthetic dataset

In [ ]:
from torch.utils.data import DataLoader

train = gl.ml.datasets.LensClassifierDataset(n_samples=400, npix=48,
                                             deltapix=0.05, seed=0)
val   = gl.ml.datasets.LensClassifierDataset(n_samples=100, npix=48,
                                             deltapix=0.05, seed=1000)
train_loader = DataLoader(train, batch_size=32, shuffle=True)
val_loader   = DataLoader(val,   batch_size=32)

# Visualize a few samples.
xs, ys = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    gl.viz.imshow_log(xs[i, 0] + xs[i, 0].min().abs() + 1e-3, ax=ax,
                      title='lens' if int(ys[i]) else 'no-lens')
plt.tight_layout(); plt.show()


## 2. Train the CNN

In [ ]:
model = gl.ml.models.LensCNN(in_channels=1, n_classes=2)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')

history = gl.ml.train.fit_model(
    model, train_loader, val_loader,
    loss_fn=nn.CrossEntropyLoss(),
    lr=1e-3, epochs=8,
    metrics={'acc': gl.ml.train.accuracy},
    log_every=1,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.train_loss, label='train')
axes[0].plot(history.val_loss, label='val')
axes[0].set(yscale='log', xlabel='epoch', ylabel='loss'); axes[0].legend()
axes[1].plot(history.metrics['acc'], label='train acc')
axes[1].plot(history.metrics['val_acc'], label='val acc')
axes[1].set(xlabel='epoch', ylabel='accuracy'); axes[1].legend()
plt.show()


## 3. Confusion matrix on a fresh test set

In [ ]:
test = gl.ml.datasets.LensClassifierDataset(n_samples=200, npix=48,
                                            deltapix=0.05, seed=7777)
test_loader = DataLoader(test, batch_size=64)

preds, labels = [], []
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        # `model` lives on `device` (MPS / CUDA / CPU); we must
        # move the test batch to the same device before the
        # forward pass — DataLoader returns CPU tensors by default.
        p = model(x.to(device)).argmax(dim=-1).cpu()
        preds.extend(p.tolist()); labels.extend(y.tolist())
preds = np.array(preds); labels = np.array(labels)
cm = np.array([[((preds==i) & (labels==j)).sum() for j in (0,1)] for i in (0,1)])
print('confusion matrix [pred x truth]:')
print(cm)
print(f'accuracy = {(preds == labels).mean():.3f}')
